# EEG_07e — Pre-costruzione Tensori Grafo / Ipergrafo

Costruisce e salva in `data/interim/graphs/` i tensori PyG `Data(x, edge_index, y, subj)`
per ogni trial, pronti per EEG_08/09/10/11.

Supporta:
- Metodi di connettività: **PCC**, **PLV**, **wPLI**
- K valori per k-NN
- **Pruning**: soglia sugli edge deboli + rimozione canali a bassa connettività
- Costruzione **ipergrafo** (per EEG_11)

**61 canali** (N_CHANS=61 — A1/A2 inclusi, Pz/POz non presenti nell'H5)

In [ ]:
# ============================================================
# CONFIGURAZIONE — modifica qui prima di eseguire
# ============================================================

# Metodi di connettività da costruire
METHODS = ["pcc"]           # "pcc" | "plv" | "wpli" — lista

# K-vicini per k-NN graph
K_VALUES = [6]              # lista di k da provare

# Soglia edge: rimuove archi con peso < threshold (0.0 = nessuna rimozione)
EDGE_THRESHOLD = 0.0

# Pruning canali: rimuove canali con connettività media < mean - sigma*std
# None = nessun pruning; es. 2.0 = rimuove canali < media - 2*std
CHANNEL_PRUNING_SIGMA = None

# Canali da escludere esplicitamente (es. ["A1", "A2"] o [])
CHANNEL_DROP_NAMES = []

# Costruisci anche ipergrafo (per EEG_11)
BUILD_HGNN = True
K_HYPER    = 6              # k per iperedge (ogni nodo ha k+1 membri)

# Forza ricostruzione anche se file già esistente
FORCE_REBUILD = True

# Schema label
CLUSTER_SCHEME = "concr4"   # "concr4" | "ward4" | "sem5" | "pos4" | "raw110"

print("Config OK")

In [ ]:
# ============================================================
# IMPORT E PATHS
# ============================================================

import os, sys
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import pandas as pd
import h5py
import torch
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from torch_geometric.data import Data

# Trova project root (dove sta .git)
project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)
sys.path.insert(0, str(project_root / "scripts"))
from utils import load_label_scheme, get_keep_channels

# Paths
META_CSV   = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH  = project_root / "data" / "interim" / "ebneuro.locs"
H5_DIR     = project_root / "data" / "processed"
INTERIM    = project_root / "data" / "interim"
GRAPHS_DIR = INTERIM / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

N_CHANS = 61

# Canali validi: tutti e 61 (A1/A2 inclusi, Pz/POz non registrati)
keep_idx, keep_names = get_keep_channels(ELOC_PATH)
assert len(keep_idx) == N_CHANS

# Schema label
labelid2cluster, N_CLASSES, cluster_names = load_label_scheme(CLUSTER_SCHEME, INTERIM)

print(f"Project root : {project_root}")
print(f"N_CHANS      : {N_CHANS}")
print(f"N_CLASSES    : {N_CLASSES} ({CLUSTER_SCHEME})")
print(f"Output dir   : {GRAPHS_DIR}")

In [ ]:
# ============================================================
# FUNZIONI DI CONNETTIVITÀ
# ============================================================

def pcc_matrix(x_np: np.ndarray) -> np.ndarray:
    """Matrice di correlazione di Pearson |PCC| tra canali. Shape: (N, N)"""
    pcc = np.abs(np.corrcoef(x_np))   # (N, N)
    np.fill_diagonal(pcc, 0.0)
    return pcc


def plv_matrix(x_np: np.ndarray) -> np.ndarray:
    """Phase Locking Value tra canali tramite Hilbert transform. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    phases = np.angle(hilbert(x_np, axis=1))   # (N, T)
    plv = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            diff = phases[i] - phases[j]
            plv[i, j] = plv[j, i] = np.abs(np.mean(np.exp(1j * diff)))
    return plv


def wpli_matrix(x_np: np.ndarray) -> np.ndarray:
    """Weighted Phase Lag Index (wPLI) tra canali. Shape: (N, N)"""
    from scipy.signal import hilbert
    N = x_np.shape[0]
    analytic = hilbert(x_np, axis=1)   # (N, T)
    wpli = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            cs = analytic[i] * np.conj(analytic[j])   # cross-spectrum
            im = np.imag(cs)
            w = np.abs(im)
            denom = np.mean(w)
            wpli[i, j] = wpli[j, i] = np.abs(np.mean(im * w)) / (denom + 1e-9)
    return wpli


def knn_edge_index(matrix: np.ndarray, k: int,
                   threshold: float = 0.0) -> torch.LongTensor:
    """k-NN graph da matrice di connettività. Restituisce edge_index (2, E)."""
    N = matrix.shape[0]
    rows, cols = [], []
    for i in range(N):
        row = matrix[i].copy()
        row[i] = -1.0
        # Applica soglia: azzera edge deboli
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k = np.argsort(row)[-k:]
        for j in top_k:
            if row[j] > 0.0 or threshold == 0.0:
                rows += [i, j]
                cols += [j, i]
    edge_index = torch.tensor([rows, cols], dtype=torch.long)
    return edge_index


def hyperedge_index(x_np: np.ndarray, k: int,
                    threshold: float = 0.0) -> torch.LongTensor:
    """
    Ipergrafo k-NN basato su PCC per-trial.
    Ogni nodo i è centro di un'iperedge che include i + top-k vicini.
    Restituisce hyperedge_index (2, E) con [vertex, edge_id].
    """
    pcc = pcc_matrix(x_np)
    N = pcc.shape[0]
    vertex_list, edge_list = [], []
    n_edges = 0
    for i in range(N):
        row = pcc[i].copy()
        if threshold > 0.0:
            row[row < threshold] = 0.0
        top_k = np.argsort(row)[-k:]
        members = [i] + [j for j in top_k if (row[j] > 0.0 or threshold == 0.0)]
        for v in members:
            vertex_list.append(v)
            edge_list.append(n_edges)
        n_edges += 1
    return torch.tensor([vertex_list, edge_list], dtype=torch.long)


print("Funzioni connettività caricate")

In [ ]:
# ============================================================
# ANALISI VISIVA — connettività media per canale
# Aiuta a decidere quali canali prunare
# ============================================================

# Carica un soggetto campione per analisi
meta = pd.read_csv(META_CSV)
sample_subj = str(meta["subject_id"].unique()[0]).zfill(3)
h5_path = H5_DIR / f"subject_{sample_subj}_preprocessed.h5"

with h5py.File(h5_path, "r") as f:
    x_all = f["eeg"][:, keep_idx, :].astype(np.float32)  # (T, 61, S)

print(f"Soggetto campione: {sample_subj}, shape: {x_all.shape}")

# Calcola PCC media su tutti i trial
print("Calcolo PCC media su tutti i trial...")
pcc_sum = np.zeros((N_CHANS, N_CHANS))
for t in range(x_all.shape[0]):
    pcc_sum += pcc_matrix(x_all[t])
pcc_avg = pcc_sum / x_all.shape[0]

# Connettività per canale = somma riga
ch_connectivity = pcc_avg.sum(axis=1)
mean_conn = ch_connectivity.mean()
std_conn  = ch_connectivity.std()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap PCC
sns.heatmap(pcc_avg, ax=axes[0], cmap="RdYlBu_r", vmin=0, vmax=0.5,
            xticklabels=keep_names, yticklabels=keep_names)
axes[0].set_title(f"PCC Media — Soggetto {sample_subj}")
axes[0].tick_params(labelsize=6)

# Connettività per canale
colors = ["red" if c < mean_conn - 2*std_conn else "steelblue" for c in ch_connectivity]
axes[1].bar(range(N_CHANS), ch_connectivity, color=colors)
axes[1].axhline(mean_conn, color="black", linestyle="--", label="Media")
axes[1].axhline(mean_conn - std_conn, color="orange", linestyle="--", label="Mean - 1σ")
axes[1].axhline(mean_conn - 2*std_conn, color="red", linestyle="--", label="Mean - 2σ")
axes[1].set_xticks(range(N_CHANS))
axes[1].set_xticklabels(keep_names, rotation=90, fontsize=7)
axes[1].set_title("Connettività per Canale (somma PCC)")
axes[1].legend()

plt.tight_layout()
fig.savefig(project_root / "figures" / f"eeg07e_channel_connectivity_subj{sample_subj}.png",
            dpi=120, bbox_inches="tight")
plt.show()

print(f"Canali sotto mean-2σ (candidati al pruning):")
for i, (name, conn) in enumerate(zip(keep_names, ch_connectivity)):
    if conn < mean_conn - 2*std_conn:
        print(f"  idx={i:2d} {name:6s} conn={conn:.4f}")

In [ ]:
# ============================================================
# PRUNING CANALI
# Applica CHANNEL_PRUNING_SIGMA o CHANNEL_DROP_NAMES
# ============================================================

pruned_idx   = list(keep_idx)   # copia
pruned_names = list(keep_names)

# Pruning per nome esplicito
if CHANNEL_DROP_NAMES:
    drop_set = set(CHANNEL_DROP_NAMES)
    pruned_idx   = [i for i, n in zip(pruned_idx, pruned_names) if n not in drop_set]
    pruned_names = [n for n in pruned_names if n not in drop_set]
    print(f"Rimossi per nome: {CHANNEL_DROP_NAMES}")

# Pruning per sigma (basato su PCC avg del soggetto campione)
if CHANNEL_PRUNING_SIGMA is not None:
    threshold_conn = mean_conn - CHANNEL_PRUNING_SIGMA * std_conn
    low_conn_names = [keep_names[i] for i in range(N_CHANS)
                      if ch_connectivity[i] < threshold_conn]
    print(f"Canali sotto mean-{CHANNEL_PRUNING_SIGMA}σ: {low_conn_names}")
    drop_set = set(low_conn_names)
    pruned_idx   = [i for i, n in zip(pruned_idx, pruned_names) if n not in drop_set]
    pruned_names = [n for n in pruned_names if n not in drop_set]

N_PRUNED = len(pruned_idx)
print(f"Canali dopo pruning: {N_PRUNED}/{N_CHANS}")
print(f"Canali mantenuti: {pruned_names[:10]}..." if N_PRUNED > 10 else f"Canali: {pruned_names}")

In [ ]:
# ============================================================
# BUILD GRAPH TENSORS (PCC / PLV / wPLI)
# Output: data/interim/graphs/graph_{method}_k{k}.pt
# ============================================================

CONN_FN = {"pcc": pcc_matrix, "plv": plv_matrix, "wpli": wpli_matrix}

for method in METHODS:
    for k in K_VALUES:
        out_path = GRAPHS_DIR / f"graph_{method}_k{k}.pt"
        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip (già esiste): {out_path.name}")
            continue

        print(f"\nCostruendo graph_{method}_k{k}...")
        conn_fn = CONN_FN[method]
        data_list = []

        for _, row in tqdm(meta.iterrows(), total=len(meta), desc=f"{method} k={k}"):
            subj_id  = str(row["subject_id"]).zfill(3)
            h5_path  = H5_DIR / f"subject_{subj_id}_preprocessed.h5"
            label_id = int(row["label_id"])

            if label_id not in labelid2cluster:
                continue
            y = labelid2cluster[label_id]

            epoch_idx = int(row["epoch_idx"])

            with h5py.File(h5_path, "r") as f:
                x_np = f["eeg"][epoch_idx, pruned_idx, :].astype(np.float32)  # (N_PRUNED, T)

            # Connettività per-trial
            matrix     = conn_fn(x_np)
            edge_index = knn_edge_index(matrix, k=k, threshold=EDGE_THRESHOLD)

            x_tensor = torch.tensor(x_np, dtype=torch.float32)
            data_list.append(Data(
                x          = x_tensor,
                edge_index = edge_index,
                y          = torch.tensor(y, dtype=torch.long),
                subj       = torch.tensor(int(row["subject_id"]), dtype=torch.long),
            ))

        torch.save(data_list, out_path)
        print(f"  Salvato: {out_path.name} — {len(data_list)} grafi, N_nodes={N_PRUNED}")

In [ ]:
# ============================================================
# BUILD HYPERGRAPH TENSORS
# Output: data/interim/graphs/hgraph_k{k}.pt
# ============================================================

if BUILD_HGNN:
    for k in K_VALUES:
        out_path = GRAPHS_DIR / f"hgraph_k{k}.pt"
        if out_path.exists() and not FORCE_REBUILD:
            print(f"Skip (già esiste): {out_path.name}")
            continue

        print(f"\nCostruendo hgraph_k{k}...")
        data_list = []

        for _, row in tqdm(meta.iterrows(), total=len(meta), desc=f"hgraph k={k}"):
            subj_id  = str(row["subject_id"]).zfill(3)
            h5_path  = H5_DIR / f"subject_{subj_id}_preprocessed.h5"
            label_id = int(row["label_id"])

            if label_id not in labelid2cluster:
                continue
            y = labelid2cluster[label_id]

            epoch_idx = int(row["epoch_idx"])

            with h5py.File(h5_path, "r") as f:
                x_np = f["eeg"][epoch_idx, pruned_idx, :].astype(np.float32)

            he_index    = hyperedge_index(x_np, k=k, threshold=EDGE_THRESHOLD)
            n_hyperedges = he_index[1].max().item() + 1

            x_tensor = torch.tensor(x_np, dtype=torch.float32)
            data_list.append(Data(
                x              = x_tensor,
                hyperedge_index = he_index,
                num_hyperedges  = n_hyperedges,
                y               = torch.tensor(y, dtype=torch.long),
                subj            = torch.tensor(int(row["subject_id"]), dtype=torch.long),
            ))

        torch.save(data_list, out_path)
        print(f"  Salvato: {out_path.name} — {len(data_list)} ipergrafi, N_nodes={N_PRUNED}")

In [ ]:
# ============================================================
# SANITY CHECK — verifica file salvati
# ============================================================

print("=== File salvati in", GRAPHS_DIR, "===")
for pt_file in sorted(GRAPHS_DIR.glob("*.pt")):
    data_list = torch.load(pt_file, weights_only=False)
    d0 = data_list[0]
    if hasattr(d0, 'edge_index'):
        print(f"  {pt_file.name}: {len(data_list)} grafi | x={d0.x.shape} | "
              f"edge_index={d0.edge_index.shape} | y={d0.y.item()} | subj={d0.subj.item()}")
    elif hasattr(d0, 'hyperedge_index'):
        print(f"  {pt_file.name}: {len(data_list)} ipergrafi | x={d0.x.shape} | "
              f"he={d0.hyperedge_index.shape} | n_he={d0.num_hyperedges} | y={d0.y.item()}")

print("\n✅ Sanity check completato")
print(f"   Canali usati: {N_PRUNED} {'(pruned)' if N_PRUNED < N_CHANS else '(tutti 61)'}")
print(f"   Schema label: {CLUSTER_SCHEME} ({N_CLASSES} classi)")